In [13]:
import os, random
import numpy as np
import torch
import torch.nn.functional as F
import tifffile as tiff
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PATCH = 128
BATCH = 4
EPOCHS = 100         
LR = 2e-4            


TRAIN_NOISY = "/content/ct_denoise/train/noisy"
TRAIN_CLEAN = "/content/ct_denoise/train/clean"

TEST_NOISY = "/content/ct_denoise/test/noisy"
TEST_CLEAN = "/content/ct_denoise/test/clean"

SAVE_PATH = "/content/ct_eformer_finetuned.pth"


def normalize_tif(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    if x.max() <= 1.0:
        return x
    if x.max() <= 255:
        return x / 255.0
    if x.max() <= 65535:
        return x / 65535.0
    return (x - x.min()) / (x.max() - x.min() + 1e-8)


class CTPairsTIF(Dataset):
    def __init__(
        self,
        noisy_dir: str,
        clean_dir: str,
        patch_size: int | None = 128,
        augment: bool = True,
    ) -> None:
        self.noisy_dir = noisy_dir
        self.clean_dir = clean_dir
        self.patch_size = patch_size
        self.augment = augment

        noisy_names = set([
            n for n in os.listdir(noisy_dir)
            if n.lower().endswith((".tif", ".tiff"))
        ])
        clean_names = set([
            n for n in os.listdir(clean_dir)
            if n.lower().endswith((".tif", ".tiff"))
        ])

        self.names = sorted(noisy_names & clean_names)

        print("pairs:", len(self.names))
        print("examples:", self.names[:5])

    def __len__(self) -> int:
        return len(self.names)

    def __getitem__(self, i: int) -> tuple[torch.Tensor, torch.Tensor]:
        name = self.names[i]

        noisy = tiff.imread(os.path.join(self.noisy_dir, name))
        clean = tiff.imread(os.path.join(self.clean_dir, name))

        noisy = normalize_tif(noisy)
        clean = normalize_tif(clean)

        h, w = noisy.shape

        if self.patch_size is not None:
            p = self.patch_size
            y = random.randint(0, h - p)
            x = random.randint(0, w - p)
            noisy = noisy[y:y+p, x:x+p]
            clean = clean[y:y+p, x:x+p]

        if self.augment:
            if random.random() < 0.5:
                noisy = np.flip(noisy, axis=1).copy()
                clean = np.flip(clean, axis=1).copy()
            if random.random() < 0.5:
                noisy = np.flip(noisy, axis=0).copy()
                clean = np.flip(clean, axis=0).copy()

        noisy = torch.from_numpy(noisy.copy()).float()[None]
        clean = torch.from_numpy(clean.copy()).float()[None]

        return noisy, clean

In [6]:
class CTPairsTIF(Dataset):
    def __init__(
        self,
        noisy_dir: str,
        clean_dir: str,
        names: list[str] | None = None, 
        patch_size: int | None = 128,
        augment: bool = True,
    ) -> None:
        self.noisy_dir = noisy_dir
        self.clean_dir = clean_dir
        self.patch_size = patch_size
        self.augment = augment

        if names is not None:
            self.names = sorted(names)
        else:
            noisy_names = set([
                n for n in os.listdir(noisy_dir)
                if n.lower().endswith((".tif", ".tiff"))
            ])
            clean_names = set([
                n for n in os.listdir(clean_dir)
                if n.lower().endswith((".tif", ".tiff"))
            ])
            self.names = sorted(noisy_names & clean_names)

        print("pairs:", len(self.names))
        print("examples:", self.names[:5])

    def __len__(self) -> int:
        return len(self.names)

    def __getitem__(self, i: int) -> tuple[torch.Tensor, torch.Tensor]:
        name = self.names[i]

        noisy = tiff.imread(os.path.join(self.noisy_dir, name))
        clean = tiff.imread(os.path.join(self.clean_dir, name))

        noisy = normalize_tif(noisy)
        clean = normalize_tif(clean)

        h, w = noisy.shape

        if self.patch_size is not None:
            p = self.patch_size
            y = random.randint(0, h - p)
            x = random.randint(0, w - p)
            noisy = noisy[y:y+p, x:x+p]
            clean = clean[y:y+p, x:x+p]

        if self.augment:
            if random.random() < 0.5:
                noisy = np.flip(noisy, axis=1).copy()
                clean = np.flip(clean, axis=1).copy()
            if random.random() < 0.5:
                noisy = np.flip(noisy, axis=0).copy()
                clean = np.flip(clean, axis=0).copy()

        noisy = torch.from_numpy(noisy.copy()).float()[None]
        clean = torch.from_numpy(clean.copy()).float()[None]

        return noisy, clean

In [9]:
# пути и разбиение
NOISY_DIR = "/home/jupyter/filestore/filestorage/V_beton30_angle05"
CLEAN_DIR = "/home/jupyter/filestore/filestorage/V_beton30_angle005"

SAVE_PATH = "/home/jupyter/Eformer/ct_eformer_finetuned.pth"

all_names = sorted(
    n for n in os.listdir(NOISY_DIR)
    if n.lower().endswith((".tif", ".tiff"))
)

n_total = len(all_names)
n_test = max(1, int(n_total * 0.1))  # 10% как test, непрерывным блоком с конца

train_names = all_names[:-n_test]
test_names = all_names[-n_test:]

print(f"total: {n_total} | train: {len(train_names)} | test: {len(test_names)}")
print("train range:", train_names[0], "...", train_names[-1])
print("test range:", test_names[0], "...", test_names[-1])

total: 2816 | train: 2535 | test: 281
train range: rec_00000.tif ... rec_02534.tif
test range: rec_02535.tif ... rec_02815.tif


In [10]:
#создание датасетов
train_ds = CTPairsTIF(NOISY_DIR, CLEAN_DIR, names=train_names, patch_size=PATCH, augment=True)
test_ds = CTPairsTIF(NOISY_DIR, CLEAN_DIR, names=test_names, patch_size=None, augment=False)

noisy, clean = train_ds[0]
print("noisy:", noisy.shape, noisy.dtype, noisy.min().item(), noisy.max().item())
print("clean:", clean.shape, clean.dtype, clean.min().item(), clean.max().item())

pairs: 2535
examples: ['rec_00000.tif', 'rec_00001.tif', 'rec_00002.tif', 'rec_00003.tif', 'rec_00004.tif']
pairs: 281
examples: ['rec_02535.tif', 'rec_02536.tif', 'rec_02537.tif', 'rec_02538.tif', 'rec_02539.tif']
noisy: torch.Size([1, 128, 128]) torch.float32 0.042221713811159134 0.3458762466907501
clean: torch.Size([1, 128, 128]) torch.float32 0.0 0.23308156430721283


In [14]:
class LearnableSobel(nn.Module):

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        gx = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32)
        gy = gx.t()
        self.gx = torch.nn.Conv2d(in_ch, in_ch, 3, padding=1, groups=in_ch, bias=False)
        self.gy = torch.nn.Conv2d(in_ch, in_ch, 3, padding=1, groups=in_ch, bias=False)
        with torch.no_grad():
            self.gx.weight.copy_(gx.expand(in_ch, 1, 3, 3))
            self.gy.weight.copy_(gy.expand(in_ch, 1, 3, 3))
        self.gx.weight.requires_grad_(True)
        self.gy.weight.requires_grad_(True)
        self.proj = torch.nn.Conv2d(in_ch * 2, out_ch, 1)

    def forward(self, x):
        ex = self.gx(x)
        ey = self.gy(x)
        edge = torch.cat([ex, ey], dim=1)
        return self.proj(edge)


def window_partition(x, win):
    B, H, W, C = x.shape
    x = x.view(B, H // win, win, W // win, win, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).reshape(-1, win, win, C)
    return windows


def window_reverse(windows, win, H, W):
    B = int(windows.shape[0] / (H * W / win / win))
    x = windows.view(B, H // win, W // win, win, win, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).reshape(B, H, W, -1)
    return x


class WindowAttention(nn.Module):
    def __init__(self, dim, win, num_heads):
        super().__init__()
        self.dim = dim
        self.win = win
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        Bw, N, C = x.shape
        qkv = self.qkv(x).reshape(Bw, N, 3, self.num_heads, C // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(Bw, N, C)
        return self.proj(out)


class LeWinBlock(nn.Module):
    def __init__(self, dim, win=8, num_heads=4, mlp_ratio=4.0):
        super().__init__()
        self.win = win
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, win, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        pad_h = (self.win - H % self.win) % self.win
        pad_w = (self.win - W % self.win) % self.win
        if pad_h or pad_w:
            x = F.pad(x, (0, pad_w, 0, pad_h))
        _, _, Hp, Wp = x.shape

        x = x.permute(0, 2, 3, 1)
        shortcut = x
        x = self.norm1(x)
        windows = window_partition(x, self.win).view(-1, self.win * self.win, C)
        attn_out = self.attn(windows)
        attn_out = attn_out.view(-1, self.win, self.win, C)
        x = window_reverse(attn_out, self.win, Hp, Wp)
        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        x = x.permute(0, 3, 1, 2)

        if pad_h or pad_w:
            x = x[:, :, :H, :W]
        return x

In [16]:
test_block = LeWinBlock(dim=32, win=8, num_heads=4).to(DEVICE)
dummy = torch.randn(2, 32, 128, 128).to(DEVICE)
out = test_block(dummy)
print("LeWinBlock output shape:", out.shape)  

test_sobel = LearnableSobel(in_ch=1, out_ch=8).to(DEVICE)
dummy_img = torch.randn(2, 1, 128, 128).to(DEVICE)
edge_out = test_sobel(dummy_img)
print("LearnableSobel output shape:", edge_out.shape)  

LeWinBlock output shape: torch.Size([2, 32, 128, 128])
LearnableSobel output shape: torch.Size([2, 8, 128, 128])


In [17]:
class LC2Down(nn.Module):
    """LeWin block(s) + edge-feature concat + conv + downsample."""

    def __init__(self, dim, out_dim, win, num_heads, n_blocks=1):
        super().__init__()
        self.blocks = nn.ModuleList([LeWinBlock(dim, win, num_heads) for _ in range(n_blocks)])
        self.sobel = LearnableSobel(1, dim // 4)
        self.fuse = nn.Conv2d(dim + dim // 4, dim, 3, padding=1)
        self.down = nn.Conv2d(dim, out_dim, 4, stride=2, padding=1)

    def forward(self, x, orig_img):
        for blk in self.blocks:
            x = blk(x)
        edge = self.sobel(orig_img)
        edge = F.interpolate(edge, size=x.shape[-2:], mode="bilinear", align_corners=False)
        x = self.fuse(torch.cat([x, edge], dim=1))
        skip = x
        x = self.down(x)
        return x, skip


class LC2Up(nn.Module):
    """Upsample + skip concat + LeWin block(s)."""

    def __init__(self, dim, out_dim, win, num_heads, n_blocks=1):
        super().__init__()
        self.up = nn.ConvTranspose2d(dim, out_dim, 4, stride=2, padding=1)
        self.fuse = nn.Conv2d(out_dim * 2, out_dim, 3, padding=1)
        self.blocks = nn.ModuleList([LeWinBlock(out_dim, win, num_heads) for _ in range(n_blocks)])

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = self.fuse(torch.cat([x, skip], dim=1))
        for blk in self.blocks:
            x = blk(x)
        return x


class Eformer(nn.Module):
    """
    Encoder-decoder на LeWin-блоках с обучаемыми Sobel-признаками на каждом
    масштабе энкодера и residual learning (сеть предсказывает шум, а не
    сразу чистое изображение).
    """

    def __init__(self, in_ch=1, base_dim=32, win=8, depths=(1, 1, 2), heads=(2, 4, 8)):
        super().__init__()
        self.input_proj = nn.Conv2d(in_ch, base_dim, 3, padding=1)

        dims = [base_dim * (2 ** i) for i in range(len(depths))]
        self.enc_stages = nn.ModuleList()
        prev = base_dim
        for i, (d, h, dim) in enumerate(zip(depths, heads, dims)):
            out_dim = dims[i + 1] if i + 1 < len(dims) else dim * 2
            self.enc_stages.append(LC2Down(prev, out_dim, win, h, n_blocks=d))
            prev = out_dim

        self.bottleneck = LeWinBlock(prev, win, heads[-1])

        self.dec_stages = nn.ModuleList()
        for i in reversed(range(len(depths))):
            skip_dim = dims[i]
            self.dec_stages.append(LC2Up(prev, skip_dim, win, heads[i], n_blocks=depths[i]))
            prev = skip_dim

        self.output_proj = nn.Conv2d(prev, in_ch, 3, padding=1)

    def forward(self, x):
        orig = x
        feat = self.input_proj(x)

        skips = []
        for stage in self.enc_stages:
            feat, skip = stage(feat, orig)
            skips.append(skip)

        feat = self.bottleneck(feat)

        for stage, skip in zip(self.dec_stages, reversed(skips)):
            feat = stage(feat, skip)

        residual = self.output_proj(feat)
        denoised = x - residual  
        return denoised

In [18]:
model = Eformer(in_ch=1, base_dim=32, win=8).to(DEVICE)
print(f"Параметров в модели: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

noisy, clean = train_ds[0]
noisy_batch = noisy.unsqueeze(0).to(DEVICE)  # [1, 1, 128, 128]

with torch.no_grad():
    out = model(noisy_batch)

print("output shape:", out.shape)  # ожидаем torch.Size([1, 1, 128, 128])
print("output min/max:", out.min().item(), out.max().item())

Параметров в модели: 3.72M
output shape: torch.Size([1, 1, 128, 128])
output min/max: 0.03861866891384125 0.6374709606170654


In [19]:
class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps = eps

    def forward(self, pred, target):
        diff = pred - target
        return torch.mean(torch.sqrt(diff * diff + self.eps * self.eps))


model = Eformer(in_ch=1, base_dim=32, win=8).to(DEVICE)
criterion = CharbonnierLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=1)


def evaluate(model, loader):
    model.eval()
    losses, psnrs, ssims = [], [], []

    with torch.no_grad():
        for noisy, clean in loader:
            noisy = noisy.to(DEVICE)
            clean = clean.to(DEVICE)

            pred = model(noisy).clamp(0, 1)

            loss = F.l1_loss(pred, clean).item()
            losses.append(loss)

            pred_np = pred[0, 0].cpu().numpy()
            clean_np = clean[0, 0].cpu().numpy()

            psnrs.append(peak_signal_noise_ratio(clean_np, pred_np, data_range=1.0))
            ssims.append(structural_similarity(clean_np, pred_np, data_range=1.0))

    model.train()
    return float(np.mean(losses)), float(np.mean(psnrs)), float(np.mean(ssims))

In [20]:
from torch.utils.data import Subset

quick_test_loader = DataLoader(Subset(test_ds, range(5)), batch_size=1, shuffle=False)

loss, psnr, ssim = evaluate(model, quick_test_loader)
print(f"loss={loss:.5f} | psnr={psnr:.2f} | ssim={ssim:.4f}")

loss=0.11003 | psnr=17.96 | ssim=0.2322


In [21]:
import time

model.train()
t0 = time.time()

for i, (noisy, clean) in enumerate(train_loader):
    if i >= 10:
        break
    noisy = noisy.to(DEVICE)
    clean = clean.to(DEVICE)

    pred = model(noisy)
    loss = criterion(pred, clean)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

dt = time.time() - t0
per_batch = dt / 10
batches_per_epoch = len(train_loader)
est_epoch_time = per_batch * batches_per_epoch

print(f"время на 10 батчей: {dt:.2f}с | ~{per_batch:.3f}с/батч")
print(f"батчей за эпоху: {batches_per_epoch}")
print(f"оценка времени на 1 эпоху: ~{est_epoch_time/60:.1f} мин")
print(f"оценка на {EPOCHS} эпох: ~{est_epoch_time*EPOCHS/3600:.2f} часов")

время на 10 батчей: 12.49с | ~1.249с/батч
батчей за эпоху: 634
оценка времени на 1 эпоху: ~13.2 мин
оценка на 100 эпох: ~22.00 часов
